In [27]:
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchRun, ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper, DuckDuckGoSearchAPIWrapper
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

import os

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

In [28]:
llm = ChatGoogleGenerativeAI(api_key=API_KEY, model="gemini-3.5-flash")
llm.invoke("salom").content[0]['text']

'Salom! Sizga qanday yordam bera olaman?'

In [29]:


@tool
def duck_duck_search(request: str):
    "     Search information on the internet using DuckDuckGo. "
    duck_search = DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper())
    return duck_search.invoke(request)


@tool
def wikipedia_search(request: str):
    "     Search information on the internet using Wikipedia. "
    wiki_search = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_search.invoke(request)


@tool
def arxiv_search(request: str):
    "     Search information on the internet using Arxiv. "
    search = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return search.invoke(request)


TOOLS = [duck_duck_search, wikipedia_search, arxiv_search]
llm_bind_tools = llm.bind_tools(TOOLS)

In [30]:
prompt = ChatPromptTemplate.from_messages([
    ('system',
     "Siz aqlli dastur yordamchisiz. Foydalanuvchi savollariga aniq javob bering. Javob uchun kerakli tool lardan foydalaning"),
    ('human', "{user_request}")
])
chain = prompt | llm_bind_tools


In [31]:
response = chain.invoke({"user_request": "2026-yilda O'zbekistonda eng so'nggi texnologik yangiliklar qanday?"})
response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'duck_duck_search', 'arguments': '{"request": "Uzbekistan technology news 2026"}'}, '__gemini_function_call_thought_signatures__': {'cehva4vv': 'EpoHCpcHARFNMg866wO+dRMPqt+clGe3ZUudEmaSPtP+cUftW3cJPX7w5+SX/A8oxUZFzE8VW2Xojp9sO1+fPmjgT5f0dHgiAn38/QBkGefPUJ0sYC5ChMRB9mHBBOtda7Icrx6JlPwv6158js0K3KyHxJKLGln95RKeriT6ZmqY/Klu+aGhPn5WOfrjEC75LtMYZx8a/zCthn3boFzSW1Drv2aC2UMlLVivQF2f/FnfHbj0vjVpAAjnhCBQkEmVrwUrxbaxAikIR5SGVAdt26y5FF7pi5y9TmsdfDqtyi8aAgTQ49BWPKm6Cfy3gYJaPEJyN2EBfJ0H/edSqog3shH3VGYS2XAkqnEQ4XZG5lvkLEUU3mWu9MEcot4VQqjUST1t4M/Ad3FUD6/sv1U8AFnWlSX/p618am/Jk0fC42Yex4d8pAT1gcjEhwaS/gyoQ+BAcQpGNIJhbVJuKv3R5zpiq4y6KWrzP8GnFnVtQVDsgitakH4Nraksp8VQbcsl6MDBmq+0+1uhtNouJF3umSQf9S7XCUuUKBRPIBvJo4Lkyr25Tme3FCc0RVWapkFRAotp6/eJKx5PFwLG1aAaJPJU92hulUINfk90ntkUsRj4kR9lL/g62RmYUiAgvAquR6ojfCMeiQA81seexIZorzR6vwX0l6o253GiBNsdrh1XP2qZaCqUkCApQVeaCN3mU1kIiIaOgwDcfyfIe0AHZexPxo3+j+pmDdzsK/VLfjEeP/m1OleZWxr31p1jU3srS09k+pwsX48BeJvm5nUs